#### Section 1: Load Dataset

In [1]:
from pyspark.sql import SparkSession
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook"

# Initialize Spark Session
spark = SparkSession.builder.appName("LightcastData").getOrCreate()

# Load Data
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine","true").option("escape", "\"").csv("data/lightcast_job_postings.csv")

# Show Schema and Sample Data
print("---This is Diagnostic check, No need to print it in the final doc---")

# df.printSchema() # comment this line when rendering the submission
df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 15:37:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


---This is Diagnostic check, No need to print it in the final doc---


26/06/14 15:38:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+-----------------+----------------------+----------+--------+---------+--------+--------------------+--------------------+--------------------+-----------+-------------------+--------------------+--------------------+---------------+----------------+--------+--------------------+-----------+-------------------+----------------+---------------------+-------------+-------------------+-------------+------------------+---------------+--------------------+--------------------+--------------------+-------------+------+-----------+----------------+-------------------+---------+-----------+--------------------+--------------------+-------------+------+--------------+-----+--------------------+-----+----------+---------------+--------------------+---------------+--------------------+------------+--------------------+------------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+------

#### Section 2: Feature Engineering

In [2]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# Drop rows with missing values in target and key features
df_clean = df.select(
    "SALARY",
    "MIN_YEARS_EXPERIENCE",
    "DURATION",
    "EMPLOYMENT_TYPE_NAME"
).dropna()

# StringIndexer for categorical variable
indexer = StringIndexer(inputCol="EMPLOYMENT_TYPE_NAME", outputCol="EMPLOYMENT_TYPE_NAME_idx")

# OneHotEncoder for categorical variable
encoder = OneHotEncoder(inputCol="EMPLOYMENT_TYPE_NAME_idx", outputCol="EMPLOYMENT_TYPE_NAME_vec")

# VectorAssembler to combine all features
assembler = VectorAssembler(
    inputCols=["MIN_YEARS_EXPERIENCE", "DURATION", "EMPLOYMENT_TYPE_NAME_vec"],
    outputCol="features"
)

# Pipeline
pipeline = Pipeline(stages=[indexer, encoder, assembler])
model = pipeline.fit(df_clean)
df_features = model.transform(df_clean)

# Show result
df_features.select("SALARY", "features").show(5, truncate=False)

+------+-------------------+
|SALARY|features           |
+------+-------------------+
|192800|[6.0,55.0,1.0,0.0] |
|125900|[12.0,18.0,1.0,0.0]|
|118560|[5.0,20.0,1.0,0.0] |
|192800|[6.0,55.0,1.0,0.0] |
|116500|[12.0,16.0,1.0,0.0]|
+------+-------------------+
only showing top 5 rows


#### Section 3: Train/Test Split

In [3]:
# Train/Test Split
# Using 80/20 split - standard split that balances having enough 
# training data while keeping sufficient data for testing
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

print(train_df.count(), test_df.count())

11604 2812


#### Section 4: Linear Regression

In [4]:
from pyspark.ml.regression import GeneralizedLinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

# Train Generalized Linear Regression model
glr = GeneralizedLinearRegression(
    family="gaussian",
    link="identity",
    featuresCol="features",
    labelCol="SALARY"
)

glr_model = glr.fit(train_df)

# Evaluate on test data
predictions = glr_model.transform(test_df)

evaluator_rmse = RegressionEvaluator(labelCol="SALARY", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="SALARY", predictionCol="prediction", metricName="r2")
evaluator_mae = RegressionEvaluator(labelCol="SALARY", predictionCol="prediction", metricName="mae")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Intercept: {glr_model.intercept:.4f}")

26/06/14 15:43:42 WARN Instrumentation: [6f288ebe] regParam is zero, which might cause numerical instability and overfitting.


R²: 0.2752
RMSE: 35410.8956
MAE: 27330.1951
Intercept: 82033.8301


In [5]:
# Extract summary statistics
summary = glr_model.summary

coefs = [glr_model.intercept] + list(glr_model.coefficients)
se = list(summary.coefficientStandardErrors)
tvals = list(summary.tValues)
pvals = list(summary.pValues)

# Diagnostic check
print("---This is Diagnostic check, No need to print it in the final doc---")
print("---The numbers below should all be same---")
print(f"Length of coefs: {len(coefs)}")
print(f"Length of se: {len(se)}")
print(f"Length of tvals: {len(tvals)}")
print(f"Length of pvals: {len(pvals)}")

# Create summary DataFrame
summary_df = pd.DataFrame({
    "Estimate": coefs,
    "Std Error": se,
    "t-stat": tvals,
    "P-Value": pvals
})

print(summary_df.to_string())

---This is Diagnostic check, No need to print it in the final doc---
---The numbers below should all be same---
Length of coefs: 5
Length of se: 5
Length of tvals: 5
Length of pvals: 5
       Estimate    Std Error     t-stat   P-Value
0  82033.830092   102.108731  66.565946  0.000000
1   6796.964269    23.512401  -0.687152  0.492001
2    -16.156599  2953.962216   0.374743  0.707859
3   1106.976203  3581.555728  -0.421511  0.673390
4  -1509.665203  2985.455723  27.477825  0.000000


#### Section 5: Diagnostic Plot

In [6]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import scipy.stats as stats

# Convert predictions to pandas
preds_pd = predictions.select("SALARY", "prediction").toPandas()
preds_pd["residuals"] = preds_pd["SALARY"] - preds_pd["prediction"]

# 2x2 Diagnostic Plot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Diagnostic Plots - Linear Regression", fontsize=16)

# 1. Predicted vs Actual
axes[0, 0].scatter(preds_pd["SALARY"], preds_pd["prediction"], alpha=0.5, color="steelblue")
axes[0, 0].set_xlabel("Actual Salary")
axes[0, 0].set_ylabel("Predicted Salary")
axes[0, 0].set_title("Predicted vs Actual")

# 2. Residuals vs Predicted
axes[0, 1].scatter(preds_pd["prediction"], preds_pd["residuals"], alpha=0.5, color="steelblue")
axes[0, 1].axhline(y=0, color="red", linestyle="--")
axes[0, 1].set_xlabel("Predicted Salary")
axes[0, 1].set_ylabel("Residuals")
axes[0, 1].set_title("Residuals vs Predicted")

# 3. Histogram of Residuals
axes[1, 0].hist(preds_pd["residuals"], bins=30, color="steelblue", edgecolor="black")
axes[1, 0].set_xlabel("Residuals")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].set_title("Histogram of Residuals")

# 4. QQ Plot
stats.probplot(preds_pd["residuals"], dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("QQ Plot of Residuals")

plt.tight_layout()
plt.savefig("_output/diagnostic_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to _output/diagnostic_plots.png")

Plot saved to _output/diagnostic_plots.png


#### Section 6: Evaluation

In [7]:
import seaborn as sns

# Evaluation Plot - Predicted vs Actual with y=x line
fig, ax = plt.subplots(figsize=(8, 6))

sns.scatterplot(data=preds_pd, x="SALARY", y="prediction", alpha=0.5, color="steelblue", ax=ax)

# Add y=x ideal fit line in red
min_val = min(preds_pd["SALARY"].min(), preds_pd["prediction"].min())
max_val = max(preds_pd["SALARY"].max(), preds_pd["prediction"].max())
ax.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", label="Ideal Fit (y=x)")

ax.set_xlabel("Actual Salary")
ax.set_ylabel("Predicted Salary")
ax.set_title("Predicted vs Actual Salary")
ax.legend()

plt.tight_layout()
plt.savefig("_output/evaluation_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

R²: 0.2752
RMSE: 35410.8956
